In [3]:
from pathlib import Path
import pandas as pd

# Locate project root from the current notebook directory
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

SPLIT_FILE = PROJECT_DIR / "data" / "splits" / "force2020_split.csv"

print("Project directory:", PROJECT_DIR)
print("Split file exists:", SPLIT_FILE.exists())

split_df = pd.read_csv(SPLIT_FILE)

print("\nColumns:", split_df.columns.tolist())
print("\nSplit counts:")
display(split_df)
print(split_df.to_string(index=False)[:3000])

Project directory: /Users/akashray/Lithology_Predictor
Split file exists: True

Columns: ['WELL', 'SPLIT']

Split counts:


,WELL,SPLIT
0,15_9-13,TRAIN
1,15_9-14,PUBLIC
2,15_9-15,TRAIN
3,15_9-17,TRAIN
4,15_9-23,BLIND_TEST
...,...,...
113,35_9-7,BLIND_TEST
114,35_9-8,PUBLIC
115,36_7-3,TRAIN
116,7_1-1,TRAIN


       WELL      SPLIT
    15_9-13      TRAIN
    15_9-14     PUBLIC
    15_9-15      TRAIN
    15_9-17      TRAIN
    15_9-23 BLIND_TEST
     16_1-2      TRAIN
   16_1-6 A      TRAIN
    16_10-1      TRAIN
    16_10-2      TRAIN
    16_10-3      TRAIN
    16_10-5      TRAIN
16_11-1 ST3      TRAIN
  16_2-11 A      TRAIN
    16_2-16      TRAIN
     16_2-6      TRAIN
     16_2-7 BLIND_TEST
     16_4-1      TRAIN
     16_5-3      TRAIN
     16_7-4      TRAIN
     16_7-5      TRAIN
     16_7-6 BLIND_TEST
     16_8-1      TRAIN
    17_11-1      TRAIN
     17_4-1 BLIND_TEST
   25_10-10     PUBLIC
    25_10-9 BLIND_TEST
   25_11-15      TRAIN
 25_11-19 S      TRAIN
   25_11-24     PUBLIC
    25_11-5      TRAIN
 25_2-13 T4      TRAIN
    25_2-14      TRAIN
     25_2-7      TRAIN
     25_3-1      TRAIN
     25_4-5      TRAIN
     25_5-1      TRAIN
     25_5-3     PUBLIC
     25_5-4      TRAIN
     25_6-1      TRAIN
     25_6-2      TRAIN
     25_6-3      TRAIN
     25_7-2      TRAIN
   25_8-5 S

In [4]:
from pathlib import Path
import shutil
import pandas as pd

# =====================================================
# 1. PATHS
# =====================================================
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw" / "FORCE_2020"
SPLIT_FILE = PROJECT_DIR / "data" / "splits" / "force2020_split.csv"
OUTPUT_DIR = PROJECT_DIR / "data" / "splits" / "wells"

# =====================================================
# 2. LOAD OFFICIAL SPLIT
# =====================================================
split_df = pd.read_csv(SPLIT_FILE)
split_df["WELL"] = split_df["WELL"].astype(str).str.strip()
split_df["SPLIT"] = split_df["SPLIT"].astype(str).str.strip().str.upper()

expected_counts = {
    "TRAIN": 98,
    "PUBLIC": 10,
    "BLIND_TEST": 10,
}

actual_counts = split_df["SPLIT"].value_counts().to_dict()
print("Official split counts:", actual_counts)

assert len(split_df) == 118, (
    f"Expected 118 manifest rows, found {len(split_df)}"
)
assert actual_counts == expected_counts, (
    f"Split counts differ from expected values: {actual_counts}"
)
assert not split_df["WELL"].duplicated().any(), "Duplicate well names in manifest"

# =====================================================
# 3. INDEX ORIGINAL LAS FILES
# =====================================================
las_files = list(RAW_DIR.rglob("*"))
las_files = [p for p in las_files if p.is_file() and p.suffix.lower() == ".las"]

def normalize(name):
    return str(name).strip().casefold()

las_lookup = {}
for path in las_files:
    key = normalize(path.stem)
    if key in las_lookup:
        raise ValueError(f"Duplicate LAS stem found: {path.stem}")
    las_lookup[key] = path

# =====================================================
# 4. VERIFY ALL WELLS BEFORE COPYING
# =====================================================
missing = [
    well for well in split_df["WELL"]
    if normalize(well) not in las_lookup
]

if missing:
    print("These manifest wells have no matching LAS file:")
    for well in missing:
        print(" -", well)
    raise FileNotFoundError(
        f"{len(missing)} wells are missing. Nothing was copied."
    )

# =====================================================
# 5. COPY INTO SEPARATE FOLDERS
# =====================================================
folder_names = {
    "TRAIN": "train",
    "PUBLIC": "public",
    "BLIND_TEST": "blind_test",
}

# Avoid silently overwriting files from an earlier run.
for folder in folder_names.values():
    target = OUTPUT_DIR / folder
    if target.exists() and any(target.iterdir()):
        raise FileExistsError(
            f"{target} already contains files. "
            "Rename/remove that output folder or choose a new OUTPUT_DIR."
        )

copied_records = []

for _, row in split_df.iterrows():
    well = row["WELL"]
    split = row["SPLIT"]

    source = las_lookup[normalize(well)]
    destination_dir = OUTPUT_DIR / folder_names[split]
    destination_dir.mkdir(parents=True, exist_ok=True)

    destination = destination_dir / source.name
    shutil.copy2(source, destination)

    copied_records.append({
        "WELL": well,
        "SPLIT": split,
        "LAS_FILE": str(destination.relative_to(PROJECT_DIR)),
    })

# Save a manifest for easy locating later.
copied_manifest = pd.DataFrame(copied_records)
copied_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)

# =====================================================
# 6. VERIFY OUTPUT
# =====================================================
print("\nCopied file counts:")
for split, folder in folder_names.items():
    files = list((OUTPUT_DIR / folder).glob("*.las"))
    print(f"{split:10s}: {len(files)} files")

print("\nOutput directory:", OUTPUT_DIR)
print("Manifest saved:", OUTPUT_DIR / "split_manifest.csv")
print("\nCompleted successfully.")

Official split counts: {'TRAIN': 98, 'PUBLIC': 10, 'BLIND_TEST': 10}

Copied file counts:
TRAIN     : 98 files
PUBLIC    : 10 files
BLIND_TEST: 10 files

Output directory: /Users/akashray/Lithology_Predictor/data/splits/wells
Manifest saved: /Users/akashray/Lithology_Predictor/data/splits/wells/split_manifest.csv

Completed successfully.


In [5]:
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

base = project_dir / "data" / "splits" / "wells"

for split in ["train", "public", "blind_test"]:
    folder = base / split
    files = list(folder.glob("*.las"))
    print(f"{split.upper():10s}: {len(files)} LAS files")

manifest = base / "split_manifest.csv"
print("\nManifest exists:", manifest.exists())

TRAIN     : 98 LAS files
PUBLIC    : 10 LAS files
BLIND_TEST: 10 LAS files

Manifest exists: True
